In [1]:
import polars as pl
from utils.general_purpose import * 
from utils.validation import *
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, accuracy_score, roc_auc_score, precision_score, recall_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer
import lightgbm as lgb

In [2]:
# VARIABLES
TRAIN_DATA_PATH = "../data/csv_files/train/"
TEST_DATA_PATH = "../data/csv_files/test/"
FEATURE_DEF_PATH = "../data/feature_definitions.csv"

# With polars
POLARS_TRAIN_DATA_PATH = "../data/train/"
POLARS_TEST_DATA_PATH = "../data/test/"

In [3]:
# Set polars config
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(-1)
# Set pandas config (the utils is made by pandas so we need this too)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [4]:
train_data = pl.read_parquet(POLARS_TRAIN_DATA_PATH+"train_data_simple_40F.parquet")
train_data.head()

sys:1: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance


firstdatedue_489D,lastrejectdate_50D,maxdpdinstldate_3546855D,dateofbirth_337D,date_decision,lastactivateddate_801D,lastapprdate_640D,datelastunpaid_3546854D,lastapplicationdate_877D,lastdelinqdate_224D,birthdate_574D,firstclxcampaign_1125D,dtlastpmtallstes_4499206D,previouscontdistrict_112M,mobilephncnt_593L,price_1097A,pmtnum_254L,isbidproduct_1095L,education_1103M,datefirstoffer_1144D,pmtssum_45A,pmtscount_423L,responsedate_4527233D,numinstunpaidmax_3546851L,requesttype_4525192L,riskassesment_302T,cntpmts24_3658933L,avgdpdtolclosure24_3658938P,assignmentdate_238D,days90_310L,lastrejectreason_759M,eir_270L,totalsettled_863A,numinstlswithdpd10_728L,lastst_736L,pctinstlsallpaidlate1d_3546856L,pmtaverage_3A,days30_165L,currdebt_22A,lastapprcommoditycat_1041M,target
cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,f64,f64,f64,bool,cat,cat,f64,f64,cat,f64,cat,cat,f64,f64,cat,f64,cat,f64,f64,f64,cat,f64,f64,f64,f64,cat,f64
"""missing""","""missing""","""missing""","""missing""","""2019-01-03""","""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""a55475b1""",1.0,25156.0,24.0,false,"""missing""","""missing""",8391.9,6.0,"""missing""",0.0,"""missing""","""missing""",10.0,0.0,"""missing""",1.0,"""a55475b1""",0.45,0.0,0.0,"""missing""",0.08333,7305.9,0.0,0.0,"""a55475b1""",0.0
"""missing""","""missing""","""missing""","""missing""","""2019-01-03""","""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""a55475b1""",1.0,25156.0,18.0,false,"""missing""","""missing""",8391.9,6.0,"""missing""",0.0,"""missing""","""missing""",10.0,0.0,"""missing""",1.0,"""a55475b1""",0.2999,0.0,0.0,"""missing""",0.08333,7305.9,0.0,0.0,"""a55475b1""",0.0
"""missing""","""2013-04-03""","""missing""","""missing""","""2019-01-04""","""missing""","""missing""","""missing""","""2013-04-03""","""missing""","""missing""","""missing""","""missing""","""a55475b1""",2.0,25156.0,36.0,false,"""missing""","""missing""",8391.9,6.0,"""missing""",0.0,"""missing""","""missing""",10.0,0.0,"""missing""",1.0,"""a55475b1""",0.45,0.0,0.0,"""D""",0.08333,7305.9,0.0,0.0,"""a55475b1""",0.0
"""missing""","""2019-01-07""","""missing""","""missing""","""2019-01-03""","""missing""","""missing""","""missing""","""2019-01-07""","""missing""","""missing""","""missing""","""missing""","""a55475b1""",1.0,25156.0,12.0,false,"""missing""","""missing""",8391.9,6.0,"""missing""",0.0,"""missing""","""missing""",10.0,0.0,"""missing""",1.0,"""P94_109_143""",0.42,0.0,0.0,"""D""",0.08333,7305.9,0.0,0.0,"""a55475b1""",0.0
"""missing""","""missing""","""missing""","""missing""","""2019-01-04""","""missing""","""missing""","""missing""","""2019-01-08""","""missing""","""missing""","""missing""","""missing""","""a55475b1""",1.0,25156.0,24.0,false,"""missing""","""missing""",8391.9,6.0,"""missing""",0.0,"""missing""","""missing""",10.0,0.0,"""missing""",1.0,"""a55475b1""",0.45,0.0,0.0,"""T""",0.08333,7305.9,0.0,0.0,"""a55475b1""",1.0


In [5]:
train_dictionary = get_feature_definitions(POLARS_TRAIN_DATA_PATH+"train_data_simple_40F.parquet", FEATURE_DEF_PATH)
train_dictionary.to_csv(FEATURE_DEF_PATH,index=False)

In [6]:
train_dictionary

,Variable,Description,Data Type
0,assignmentdate_238D,Tax authority data - date of assignment.,"dictionary<values=string, indices=int32, ordered=0>"
1,avgdpdtolclosure24_3658938P,"Average DPD (days past due) with tolerance within the past 24 months from the maximum closure date, assuming that the contract is finished. If the contract is ongoing, the calculation is based on the current date.",double
2,birthdate_574D,Client's date of birth (credit bureau data).,"dictionary<values=string, indices=int32, ordered=0>"
3,cntpmts24_3658933L,Number of months with any incoming payment in last 24 months.,double
4,currdebt_22A,Current debt amount of the client.,double
5,datefirstoffer_1144D,Date of first customer relationship management (CRM) offer.,"dictionary<values=string, indices=int32, ordered=0>"
6,datelastunpaid_3546854D,Date of the last unpaid instalment.,"dictionary<values=string, indices=int32, ordered=0>"
7,dateofbirth_337D,Client's date of birth.,"dictionary<values=string, indices=int32, ordered=0>"
8,days30_165L,Number of credit bureau queries for the last 30 days.,double
9,days90_310L,Number of credit bureau queries for the last 90 days.,double


In [7]:
stream_layer_feature = [
    'eir_270L',
    'isbidproduct_1095L',
    'mobilephncnt_593L',
    'days30_165L',
    'days90_310L',
    'requesttype_4525192L',
    'responsedate_4527233D',
    'riskassesment_302T',
    'currdebt_22A',
    'price_1097A',
    'cntpmts24_3658933L',
    'pmtaverage_3A',
    'pmtnum_254L',
    'pmtscount_423L',
    'pmtssum_45A',
    'totalsettled_863A'
]

batch_layer_feature = [
    'assignmentdate_238D',
    'birthdate_574D',
    'dateofbirth_337D',
    'education_1103M',
    'datefirstoffer_1144D',
    'firstclxcampaign_1125D',
    'lastactivateddate_801D',
    'lastapplicationdate_877D',
    'lastapprcommoditycat_1041M',
    'lastapprdate_640D',
    'lastrejectdate_50D',
    'lastrejectreason_759M',
    'lastst_736L',
    'previouscontdistrict_112M',
    'datelastunpaid_3546854D',
    'dtlastpmtallstes_4499206D',
    'maxdpdinstldate_3546855D',
    'lastdelinqdate_224D',
    'firstdatedue_489D',
    'avgdpdtolclosure24_3658938P',
    'numinstlswithdpd10_728L',
    'numinstunpaidmax_3546851L',
    'pctinstlsallpaidlate1d_3546856L'
]

In [8]:
train_data_stream = train_data.select(stream_layer_feature)
train_data_stream.head()

eir_270L,isbidproduct_1095L,mobilephncnt_593L,days30_165L,days90_310L,requesttype_4525192L,responsedate_4527233D,riskassesment_302T,currdebt_22A,price_1097A,cntpmts24_3658933L,pmtaverage_3A,pmtnum_254L,pmtscount_423L,pmtssum_45A,totalsettled_863A
f64,bool,f64,f64,f64,cat,cat,cat,f64,f64,f64,f64,f64,f64,f64,f64
0.45,false,1.0,0.0,1.0,"""missing""","""missing""","""missing""",0.0,25156.0,10.0,7305.9,24.0,6.0,8391.9,0.0
0.2999,false,1.0,0.0,1.0,"""missing""","""missing""","""missing""",0.0,25156.0,10.0,7305.9,18.0,6.0,8391.9,0.0
0.45,false,2.0,0.0,1.0,"""missing""","""missing""","""missing""",0.0,25156.0,10.0,7305.9,36.0,6.0,8391.9,0.0
0.42,false,1.0,0.0,1.0,"""missing""","""missing""","""missing""",0.0,25156.0,10.0,7305.9,12.0,6.0,8391.9,0.0
0.45,false,1.0,0.0,1.0,"""missing""","""missing""","""missing""",0.0,25156.0,10.0,7305.9,24.0,6.0,8391.9,0.0


In [9]:
train_data_stream.describe()

statistic,eir_270L,isbidproduct_1095L,mobilephncnt_593L,days30_165L,days90_310L,requesttype_4525192L,responsedate_4527233D,riskassesment_302T,currdebt_22A,price_1097A,cntpmts24_3658933L,pmtaverage_3A,pmtnum_254L,pmtscount_423L,pmtssum_45A,totalsettled_863A
str,f64,f64,f64,f64,f64,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",1.526659e6,1.526659e6,1.526659e6,1.526659e6,1.526659e6,"""1526659""","""1526659""","""1526659""",1.526659e6,1.526659e6,1.526659e6,1.526659e6,1.526659e6,1.526659e6,1.526659e6,1.526659e6
"""null_count""",0.0,0.0,0.0,0.0,0.0,"""0""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",0.292537,0.115684,1.772637,0.469904,1.191898,null,null,null,19682.37029,33103.741678,10.440284,7493.752196,16.924658,5.939719,10195.357158,92238.021942
"""std""",0.189229,null,1.13304,0.869727,1.578814,null,null,null,50835.226097,32031.689834,6.59768,1802.759697,9.679739,2.541785,11337.388871,162335.592843
"""min""",0.0,0.0,0.0,0.0,0.0,null,null,null,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0
"""25%""",0.0,null,1.0,0.0,0.0,null,null,null,0.0,15500.0,6.0,7305.9,12.0,6.0,8391.9,0.0
"""50%""",0.4005,null,1.0,0.0,1.0,null,null,null,0.0,25156.0,10.0,7305.9,12.0,6.0,8391.9,35977.664
"""75%""",0.433,null,2.0,1.0,2.0,null,null,null,13492.0,40156.0,13.0,7305.9,24.0,6.0,8391.9,118815.4
"""max""",0.45,1.0,23.0,22.0,41.0,null,null,null,1210629.1,761867.44,25.0,145257.4,60.0,121.0,476843.4,4.8035036e7


In [10]:
train_data_batch = train_data.select(batch_layer_feature)
train_data_batch.head()

assignmentdate_238D,birthdate_574D,dateofbirth_337D,education_1103M,datefirstoffer_1144D,firstclxcampaign_1125D,lastactivateddate_801D,lastapplicationdate_877D,lastapprcommoditycat_1041M,lastapprdate_640D,lastrejectdate_50D,lastrejectreason_759M,lastst_736L,previouscontdistrict_112M,datelastunpaid_3546854D,dtlastpmtallstes_4499206D,maxdpdinstldate_3546855D,lastdelinqdate_224D,firstdatedue_489D,avgdpdtolclosure24_3658938P,numinstlswithdpd10_728L,numinstunpaidmax_3546851L,pctinstlsallpaidlate1d_3546856L
cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,f64,f64,f64,f64
"""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""a55475b1""","""missing""","""missing""","""a55475b1""","""missing""","""a55475b1""","""missing""","""missing""","""missing""","""missing""","""missing""",0.0,0.0,0.0,0.08333
"""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""a55475b1""","""missing""","""missing""","""a55475b1""","""missing""","""a55475b1""","""missing""","""missing""","""missing""","""missing""","""missing""",0.0,0.0,0.0,0.08333
"""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""2013-04-03""","""a55475b1""","""missing""","""2013-04-03""","""a55475b1""","""D""","""a55475b1""","""missing""","""missing""","""missing""","""missing""","""missing""",0.0,0.0,0.0,0.08333
"""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""2019-01-07""","""a55475b1""","""missing""","""2019-01-07""","""P94_109_143""","""D""","""a55475b1""","""missing""","""missing""","""missing""","""missing""","""missing""",0.0,0.0,0.0,0.08333
"""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""2019-01-08""","""a55475b1""","""missing""","""missing""","""a55475b1""","""T""","""a55475b1""","""missing""","""missing""","""missing""","""missing""","""missing""",0.0,0.0,0.0,0.08333


## Core data (the one that go in the pipeline)
eir_270L,Interest rate.,double

isbidproduct_1095L,Flag indicating if the product is a cross-sell.,bool

currdebt_22A,Current debt amount of the client.,double

price_1097A,Credit price.,double

requesttype_4525192L,Tax authority request type.,"dictionary<values=string, indices=int32, ordered=0>"

responsedate_4527233D,Tax authority's response date.,"dictionary<values=string, indices=int32, ordered=0>"


mobilephncnt_593L,Number of persons with the same mobile phone number.,double

## Features made in the stream (derived/computed in real time)

days30_165L,Number of credit bureau queries for the last 30 days.,double

days90_310L,Number of credit bureau queries for the last 90 days.,double

cntpmts24_3658933L,Number of months with any incoming payment in last 24 months.,double

pmtaverage_3A,Average of tax deductions.,double

pmtnum_254L,Total number of loan payments made by the client.,double

pmtscount_423L,Number of tax deduction payments.,double

pmtssum_45A,Sum of tax deductions for the client.,double

totalsettled_863A,Sum of all payments made by the client.,double

riskassesment_302T,Estimated probability that the client will default on their credit obligation within the next year.,"dictionary<values=string, indices=int32, ordered=0>"

## Data Made During Batch Processing: 

avgdpdtolclosure24_3658938P,"Average DPD (days past due) with tolerance within the past 24 months from the maximum closure date, assuming that the contract is finished. If the contract is ongoing, the calculation is based on the current date.",double

numinstlswithdpd10_728L,Number of instalments that were overdue for 10 or more days.,double

numinstunpaidmax_3546851L,Maximum number of unpaid instalments.,double

pctinstlsallpaidlate1d_3546856L,Percentage of installments that are paid 1 or more days after the due date.,double

assignmentdate_238D,Tax authority data - date of assignment.,"dictionary<values=string, indices=int32, ordered=0>"

dateofbirth_337D,Client's date of birth.,"dictionary<values=string, indices=int32, ordered=0>"

birthdate_574D,Client's date of birth (credit bureau data).,"dictionary<values=string, indices=int32, ordered=0>"

education_1103M,Level of education of the client provided by external source.,"dictionary<values=string, indices=int32, ordered=0>"

datefirstoffer_1144D,Date of first customer relationship management (CRM) offer.,"dictionary<values=string, indices=int32, ordered=0>"

firstclxcampaign_1125D,Date of the client's first campaign.,"dictionary<values=string, indices=int32, ordered=0>"

lastactivateddate_801D,Contract activation date for previous applications.,"dictionary<values=string, indices=int32, ordered=0>"

lastapplicationdate_877D,Date of previous customer's application.,"dictionary<values=string, indices=int32, ordered=0>"

lastapprcommoditycat_1041M,Commodity category of the last loan applications made by the applicant.,"dictionary<values=string, indices=int32, ordered=0>"

lastapprdate_640D,Date of approval on client's most recent previous application.,"dictionary<values=string, indices=int32, ordered=0>"

lastrejectdate_50D,Date of most recent rejected application by the applicant.,"dictionary<values=string, indices=int32, ordered=0>"

lastrejectreason_759M,Reason for rejection on the most recent rejected application.,"dictionary<values=string, indices=int32, ordered=0>"

lastst_736L,Status of the client's previous credit application.,"dictionary<values=string, indices=int32, ordered=0>"

previouscontdistrict_112M,Contact district of the client's previous approved application.,"dictionary<values=string, indices=int32, ordered=0>"

datelastunpaid_3546854D,Date of the last unpaid instalment.,"dictionary<values=string, indices=int32, ordered=0>"

dtlastpmtallstes_4499206D,Date of last payment made by the applicant.,"dictionary<values=string, indices=int32, ordered=0>"

maxdpdinstldate_3546855D,Date of instalment on which client was most days past due.,"dictionary<values=string, indices=int32, ordered=0>"

firstdatedue_489D,Date of the first due date.,"dictionary<values=string, indices=int32, ordered=0>"


In [11]:
new_names = {col: col.split('_',1)[0] for col in train_data.columns}
train_data = train_data.rename(new_names)
train_data.head()

firstdatedue,lastrejectdate,maxdpdinstldate,dateofbirth,date,lastactivateddate,lastapprdate,datelastunpaid,lastapplicationdate,lastdelinqdate,birthdate,firstclxcampaign,dtlastpmtallstes,previouscontdistrict,mobilephncnt,price,pmtnum,isbidproduct,education,datefirstoffer,pmtssum,pmtscount,responsedate,numinstunpaidmax,requesttype,riskassesment,cntpmts24,avgdpdtolclosure24,assignmentdate,days90,lastrejectreason,eir,totalsettled,numinstlswithdpd10,lastst,pctinstlsallpaidlate1d,pmtaverage,days30,currdebt,lastapprcommoditycat,target
cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,cat,f64,f64,f64,bool,cat,cat,f64,f64,cat,f64,cat,cat,f64,f64,cat,f64,cat,f64,f64,f64,cat,f64,f64,f64,f64,cat,f64
"""missing""","""missing""","""missing""","""missing""","""2019-01-03""","""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""a55475b1""",1.0,25156.0,24.0,false,"""missing""","""missing""",8391.9,6.0,"""missing""",0.0,"""missing""","""missing""",10.0,0.0,"""missing""",1.0,"""a55475b1""",0.45,0.0,0.0,"""missing""",0.08333,7305.9,0.0,0.0,"""a55475b1""",0.0
"""missing""","""missing""","""missing""","""missing""","""2019-01-03""","""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""missing""","""a55475b1""",1.0,25156.0,18.0,false,"""missing""","""missing""",8391.9,6.0,"""missing""",0.0,"""missing""","""missing""",10.0,0.0,"""missing""",1.0,"""a55475b1""",0.2999,0.0,0.0,"""missing""",0.08333,7305.9,0.0,0.0,"""a55475b1""",0.0
"""missing""","""2013-04-03""","""missing""","""missing""","""2019-01-04""","""missing""","""missing""","""missing""","""2013-04-03""","""missing""","""missing""","""missing""","""missing""","""a55475b1""",2.0,25156.0,36.0,false,"""missing""","""missing""",8391.9,6.0,"""missing""",0.0,"""missing""","""missing""",10.0,0.0,"""missing""",1.0,"""a55475b1""",0.45,0.0,0.0,"""D""",0.08333,7305.9,0.0,0.0,"""a55475b1""",0.0
"""missing""","""2019-01-07""","""missing""","""missing""","""2019-01-03""","""missing""","""missing""","""missing""","""2019-01-07""","""missing""","""missing""","""missing""","""missing""","""a55475b1""",1.0,25156.0,12.0,false,"""missing""","""missing""",8391.9,6.0,"""missing""",0.0,"""missing""","""missing""",10.0,0.0,"""missing""",1.0,"""P94_109_143""",0.42,0.0,0.0,"""D""",0.08333,7305.9,0.0,0.0,"""a55475b1""",0.0
"""missing""","""missing""","""missing""","""missing""","""2019-01-04""","""missing""","""missing""","""missing""","""2019-01-08""","""missing""","""missing""","""missing""","""missing""","""a55475b1""",1.0,25156.0,24.0,false,"""missing""","""missing""",8391.9,6.0,"""missing""",0.0,"""missing""","""missing""",10.0,0.0,"""missing""",1.0,"""a55475b1""",0.45,0.0,0.0,"""T""",0.08333,7305.9,0.0,0.0,"""a55475b1""",1.0


In [13]:
train_data.slice(0, 1500).write_csv(TRAIN_DATA_PATH+"train_data_sample_40F_1500records.csv")

In [15]:
train_data = train_data.slice(0, 1500).to_pandas()
train_data.head()

,firstdatedue,lastrejectdate,maxdpdinstldate,dateofbirth,date,lastactivateddate,lastapprdate,datelastunpaid,lastapplicationdate,lastdelinqdate,birthdate,firstclxcampaign,dtlastpmtallstes,previouscontdistrict,mobilephncnt,price,pmtnum,isbidproduct,education,datefirstoffer,pmtssum,pmtscount,responsedate,numinstunpaidmax,requesttype,riskassesment,cntpmts24,avgdpdtolclosure24,assignmentdate,days90,lastrejectreason,eir,totalsettled,numinstlswithdpd10,lastst,pctinstlsallpaidlate1d,pmtaverage,days30,currdebt,lastapprcommoditycat,target
0,missing,missing,missing,missing,2019-01-03,missing,missing,missing,missing,missing,missing,missing,missing,a55475b1,1.0,25156.0,24.0,False,missing,missing,8391.9,6.0,missing,0.0,missing,missing,10.0,0.0,missing,1.0,a55475b1,0.4500,0.0,0.0,missing,0.08333,7305.9,0.0,0.0,a55475b1,0.0
1,missing,missing,missing,missing,2019-01-03,missing,missing,missing,missing,missing,missing,missing,missing,a55475b1,1.0,25156.0,18.0,False,missing,missing,8391.9,6.0,missing,0.0,missing,missing,10.0,0.0,missing,1.0,a55475b1,0.2999,0.0,0.0,missing,0.08333,7305.9,0.0,0.0,a55475b1,0.0
2,missing,2013-04-03,missing,missing,2019-01-04,missing,missing,missing,2013-04-03,missing,missing,missing,missing,a55475b1,2.0,25156.0,36.0,False,missing,missing,8391.9,6.0,missing,0.0,missing,missing,10.0,0.0,missing,1.0,a55475b1,0.4500,0.0,0.0,D,0.08333,7305.9,0.0,0.0,a55475b1,0.0
3,missing,2019-01-07,missing,missing,2019-01-03,missing,missing,missing,2019-01-07,missing,missing,missing,missing,a55475b1,1.0,25156.0,12.0,False,missing,missing,8391.9,6.0,missing,0.0,missing,missing,10.0,0.0,missing,1.0,P94_109_143,0.4200,0.0,0.0,D,0.08333,7305.9,0.0,0.0,a55475b1,0.0
4,missing,missing,missing,missing,2019-01-04,missing,missing,missing,2019-01-08,missing,missing,missing,missing,a55475b1,1.0,25156.0,24.0,False,missing,missing,8391.9,6.0,missing,0.0,missing,missing,10.0,0.0,missing,1.0,a55475b1,0.4500,0.0,0.0,T,0.08333,7305.9,0.0,0.0,a55475b1,1.0


# 1. Mimic the Streaming Pipeline

## Import pydantic class

In [20]:
from pydantic import BaseModel, Field
from typing import Optional
from datetime import datetime 

class LoanApplication(BaseModel):
    eir: float                                # Interest rate
    isbidproduct: bool                        # Is cross-sell product
    currdebt: float                           # Current debt
    price: float                              # Credit price
    requesttype: str                          # Tax authority request type
    responsedate: Optional[datetime] = None   # Tax authority response date
    mobilephncnt: float                       # Shared mobile phone count


In [21]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

# Simulate core table: loan_applications
n_rows = 5
core_data = pd.DataFrame({
    "user_id": [f"user_{i}" for i in range(n_rows)],
    "eir": np.round(np.random.uniform(0.03, 0.08, n_rows), 4),
    "isbidproduct": np.random.choice([True, False], n_rows),
    "currdebt": np.round(np.random.uniform(5000, 20000, n_rows), 2),
    "price": np.round(np.random.uniform(10000, 30000, n_rows), 2),
    "requesttype": np.random.choice(["type_a", "type_b", "type_c"], n_rows),
    "responsedate": [(datetime.today() - timedelta(days=np.random.randint(10, 100))).strftime('%Y-%m-%d') for _ in range(n_rows)],
    "mobilephncnt": np.random.randint(1, 5, n_rows),
    "date": [(datetime.today() - timedelta(days=np.random.randint(1, 30))).strftime('%Y-%m-%d') for _ in range(n_rows)]
})

# Simulate streaming enrichment table: credit_bureau_agg
credit_bureau_data = pd.DataFrame({
    "user_id": core_data["user_id"],
    "days30": np.random.randint(0, 10, n_rows),
    "days90": np.random.randint(5, 20, n_rows),
    "cntpmts24": np.random.randint(5, 24, n_rows),
    "pmtaverage": np.round(np.random.uniform(500, 2000, n_rows), 2),
    "pmtscount": np.random.randint(1, 12, n_rows),
    "totalsettled": np.round(np.random.uniform(3000, 25000, n_rows), 2),
    "riskassesment": np.random.choice(["low", "medium", "high"], n_rows)
})

# Simulate batch feature table: contract_history_agg
contract_data = pd.DataFrame({
    "user_id": core_data["user_id"],
    "avgdpdtolclosure24": np.round(np.random.uniform(0, 1, n_rows), 4),
    "numinstlswithdpd10": np.random.randint(0, 10, n_rows),
    "numinstunpaidmax": np.random.randint(0, 5, n_rows),
    "pctinstlsallpaidlate1d": np.round(np.random.uniform(0, 1, n_rows), 4),
    "assignmentdate": [(datetime.today() - timedelta(days=np.random.randint(100, 1000))).strftime('%Y-%m-%d') for _ in range(n_rows)],
    "dateofbirth": [(datetime.today() - timedelta(days=365*25 + np.random.randint(0, 3000))).strftime('%Y-%m-%d') for _ in range(n_rows)],
    "education": np.random.choice(["highschool", "bachelor", "master", "phd"], n_rows),
    "lastrejectreason": np.random.choice(["low_income", "bad_history", "incomplete_docs"], n_rows),
    "lastst": np.random.choice(["A", "B", "C", "D"], n_rows),
    "lastapprcommoditycat": np.random.choice(["cat_a", "cat_b", "cat_c"], n_rows)
})

In [ ]:
# mimic the schema registry
schema = list(train_data.columns)

# mimic the streaming pipeline (process row by row)
for idx, row in train_data.iterrows():
    event = row.to_dict()